# Exercise 1a (Text summarization)

### First steps

### Pegasus = Encoder-Decoder

Pegasus uses 'abstractive' generation. This means, it understands meaning and uses own words for summarization (based on training knowledge).

Pegasus is known to be hallucination-prone because it was trained on BBC articles with punchy and sometimes inaccurate first sentences.

Gap Sentence Generation (GSG): During its initial training, entire "important" sentences were removed from documents. The model had to "fill in the gaps" by generating those missing sentences based on the remaining context. This forced it to learn how to identify and synthesize the most critical information.

Extreme Fine-tuning: The -xsum version is fine-tuned on BBC news articles where the goal is to create a "one-sentence" summary that answers "What is this article about?" rather than just repeating the lead sentence.

#### Strenghts

- True abstraction
- Extreme summarizations

#### Weaknesses

- Hallucinations
- Heavy bias toward producing single sentence outputs
- Relatively large (568 million params)

In [2]:
from transformers import pipeline

summarizer = pipeline("summarization", model="google/pegasus-xsum")

text = """
The NASA Mars 2020 Perseverance rover is exploring Jezero Crater.
It is looking for signs of ancient life and collecting samples of rock
and regolith (broken rock and soil) for possible return to Earth.
"""

summary = summarizer(text, max_length=60, min_length=10, do_sample=False)

print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0
Your max_length is set to 60, but your input_length is only 44. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=22)


The US space agency has sent a robot rover to the surface of Mars for the first time.


### Overwhelming the transformer with sequence length and equations

In [4]:
encode_decode_attention_text = """
3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512. Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization. We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i.

3.2 Attention An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum 3 Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key. 3.2.1 Scaled Dot-Product Attention We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the query with all keys, divide each by √ dk, and apply a softmax function to obtain the weights on the values. In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix Q. The keys and values are also packed together into matrices K and V . We compute the matrix of outputs as: Attention(Q, K, V ) = softmax(QKT √ dk )V (1) The two most commonly used attention functions are additive attention [2], and dot-product (multiplicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor of √ 1 dk . Additive attention computes the compatibility function using a feed-forward network with a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is much faster and more space-efficient in practice, since it can be implemented using highly optimized matrix multiplication code. While for small values of dk the two mechanisms perform similarly, additive attention outperforms dot product attention without scaling for larger values of dk [3]. We suspect that for large values of dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients 4 . To counteract this effect, we scale the dot products by √ 1 dk . 3.2.2 Multi-Head Attention Instead of performing a single attention function with dmodel-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional 4To illustrate why the dot products get large, assume that the components of q and k are independent random variables with mean 0 and variance 1. Then their dot product, q · k = Pdk i=1 qiki, has mean 0 and variance dk. 4 output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this. MultiHead(Q, K, V ) = Concat(head1, ..., headh)WO where headi = Attention(QWQ i , KW K i , V WV i ) Where the projections are parameter matrices W Q i ∈ R dmodel×dk , W K i ∈ R dmodel×dk , WV i ∈ R dmodel×dv and WO ∈ R hdv×dmodel . In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [38, 2, 9]. • The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder. • Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections. See Figure 2.
"""

In [5]:
summary = summarizer(encode_decode_attention_text, max_length=60, min_length=10, do_sample=False)
print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0
Token indices sequence length is longer than the specified maximum sequence length for this model (1331 > 512). Running this sequence through the model will result in indexing errors


In this paper, we present a novel multi-head self-attention encoder and decoder.


In [14]:
summary = summarizer(encode_decode_attention_text, max_length=60, min_length=10, do_sample=False, truncation=True)
print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


In this paper, we describe two encoders and two decoders, each with its own self-attention mechanism.


### Stripping the text

In [6]:
encode_decode_attention_text_stripped = """
Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. We employ a residual connection around each of the two sub-layers, followed by layer normalization. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512. Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization. We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i. An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key. We call our particular attention "Scaled Dot-Product Attention". The input consists of queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the query with all keys, divide each by √ dk, and apply a softmax function to obtain the weights on the values. In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix Q. The keys and values are also packed together into matrices K and V. The two most commonly used attention functions are additive attention, and dot-product (multiplicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor of √ 1 dk . Additive attention computes the compatibility function using a feed-forward network with a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is much faster and more space-efficient in practice, since it can be implemented using highly optimized matrix multiplication code. While for small values of dk the two mechanisms perform similarly, additive attention outperforms dot product attention without scaling for larger values of dk. We suspect that for large values of dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients. To counteract this effect, we scale the dot products by √ 1 dk. Instead of performing a single attention function with dmodel-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this. In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. The Transformer uses multi-head attention in three different ways: In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as. The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder. Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections.
"""

In [10]:
summarizer = pipeline("summarization", model="google/pegasus-xsum")
summary = summarizer(encode_decode_attention_text_stripped, max_length=60, min_length=10, do_sample=False)
print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0
Token indices sequence length is longer than the specified maximum sequence length for this model (1032 > 512). Running this sequence through the model will result in indexing errors


We present a multi-dimensional self-attention encoder and decoder.


In [11]:
summarizer = pipeline("summarization", model="google/pegasus-xsum")

summary = summarizer(
    encode_decode_attention_text_stripped,
    max_length=60,
    min_length=20,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    no_repeat_ngram_size=3
)

print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0
Token indices sequence length is longer than the specified maximum sequence length for this model (1032 > 512). Running this sequence through the model will result in indexing errors


We present a novel encoder and decoder that can be used to perform self-attention functions on large sets of data.


### Trying the large pegagus version

In [12]:
summarizer = pipeline("summarization", model="google/pegasus-large")

summary = summarizer(
    encode_decode_attention_text,
    max_length=60,
    min_length=20,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    no_repeat_ngram_size=3
)

print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0
Token indices sequence length is longer than the specified maximum sequence length for this model (1330 > 1024). Running this sequence through the model will result in indexing errors


The output is computed as a weighted sum 3 Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled dot-product Attention. (right) Multi-head Attention consists of several attention layers running in parallel.


In [9]:
summarizer = pipeline("summarization", model="google/pegasus-large")

summary = summarizer(
    encode_decode_attention_text_stripped,
    max_length=60,
    min_length=20,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    no_repeat_ngram_size=3
)

print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0
Token indices sequence length is longer than the specified maximum sequence length for this model (1031 > 1024). Running this sequence through the model will result in indexing errors


We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than


### Splitting the text

In [16]:
encode_decode_attention_text_stripped1 = """
Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. We employ a residual connection around each of the two sub-layers, followed by layer normalization. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512. Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization. We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i. An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key. We call our particular attention "Scaled Dot-Product Attention". """

encode_decode_attention_text_stripped2 = """
The input consists of queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the query with all keys, divide each by √ dk, and apply a softmax function to obtain the weights on the values. In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix Q. The keys and values are also packed together into matrices K and V. The two most commonly used attention functions are additive attention, and dot-product (multiplicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor of √ 1 dk . Additive attention computes the compatibility function using a feed-forward network with a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is much faster and more space-efficient in practice, since it can be implemented using highly optimized matrix multiplication code. While for small values of dk the two mechanisms perform similarly, additive attention outperforms dot product attention without scaling for larger values of dk. We suspect that for large values of dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients. To counteract this effect, we scale the dot products by √ 1 dk. Instead of performing a single attention function with dmodel-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections to dk, dk and dv dimensions, respectively."""

encode_decode_attention_text_stripped3 = """
On each of these projected versions of queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this. In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. The Transformer uses multi-head attention in three different ways: In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as. The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder. Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections.
"""

In [17]:
summarizer = pipeline("summarization", model="google/pegasus-xsum")
summary = summarizer(encode_decode_attention_text_stripped1, max_length=60, min_length=10)
print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


We present a novel encoder and decoder, both with self-attention mechanisms.


In [18]:
summary = summarizer(encode_decode_attention_text_stripped2, max_length=60, min_length=10)
print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


In this paper, we present a new attention function for a d-dimensional model.


In [19]:
summary = summarizer(encode_decode_attention_text_stripped3, max_length=60, min_length=10)
print(summary[0]['summary_text'])

In our work we use the Transformer model to generate a sequence of queries, keys and values which are then projected into the model.


# Question Answering

### Distilbert = Encoder only

Smaller, faster version of BERT (Bidirectional Encoder Representations from Transformers)

A BERT model was used to distill a smaller model out of it.

#### BERT = Encoder Decoder

Is able to read whole sentences simultaneously.
Hiding words in training texts (= masking) was used in training).

Sentiment Analysis: Is this review angry or happy?
Named Entity Recognition (NER): Which words are names of people or cities?
Question Answering: Finding the exact sentence in a paragraph that answers a user's question.

#### Strengths

- Fast and efficient

#### Weaknesses

- No 'Next Sentence Prediction'
- Not generative

### Halucinations

In [29]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')
result = question_answerer(question="What was the research objective?", context=encode_decode_attention_text_stripped)
print(result)

Device set to use mps:0


{'score': 0.10681694000959396, 'start': 4851, 'end': 4888, 'answer': 'preserve the auto-regressive property'}


In [23]:
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')
result = question_answerer(question="How much is one plus one?", context=encode_decode_attention_text_stripped)
print(result)

Device set to use mps:0


{'score': 0.011308356188237667, 'start': 4408, 'end': 4468, 'answer': 'all of the keys, values and queries come from the same place'}


### Meaningful questions

In [20]:
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')
result = question_answerer(question="What is the name of the particular attention function used?", context=encode_decode_attention_text_stripped)
print(result)

Device set to use mps:0


{'score': 0.8382486999034882, 'start': 1705, 'end': 1733, 'answer': 'Scaled Dot-Product Attention'}


In [22]:
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')
result = question_answerer(question="What model architecture uses multi-head attention in three different ways?", context=encode_decode_attention_text_stripped)
print(result)

Device set to use mps:0


{'score': 0.8213588361104485, 'start': 3919, 'end': 3934, 'answer': 'The Transformer'}


# Text Generation

### FLAN-T5 = Encoder-Decoder

Thus, FLAN-T5 can understand and write.

FLAN (Fine-tuned LAnguage Net): Instead of just learning from raw text, the model was "instruction-tuned"

T5 (Text-to-Text Transfer Transformer): T5 treats every NLP problem as a "text-to-text" task. Whether it's classification or translation, the input is text and the output is always a new string of text.

"echoing" is a known quirk of T5

In [53]:
from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-base")
response = generator("Why are bananas curved?", max_length=100)
print(response[0]['generated_text'])

Device set to use mps:0
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


they are curved to make them easier to eat.


In [43]:
response = generator("Explain the scientific biological reason why bananas are curved.", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


curved bananas are a type of fruit.


In [35]:
response = generator("What is a transformer architecture in machine learning", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A transformer architecture is a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine learning architecture with a machine learning architecture that combines a machine l

In [34]:
response = generator("What is a transformer architecture in machine learning?", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


a neural network


In [41]:
response = generator("How much is one plus one", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


one plus one is one plus one.


### Flan-T5 strengths (sentiment analysis and translations)

In [44]:
response = generator("Review: This movie was a complete waste of time. Is this review positive or negative?", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


negative


In [45]:
response = generator("Translate to Spanish: Where is the nearest library?", max_length=100)
print(response[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dónde está la biblioteca más cercana?


### FLAN-T5 research expansion

In [58]:
response = generator("""Input: Explain the Multi-Head Attention layer.
Output: It allows the model to focus on different parts of the input sequence simultaneously by using multiple sets of weights.
Input: Explain the Position-wise Feed-Forward layer.
Output:""", max_new_tokens=100,
    repetition_penalty=1.2,
    do_sample=True,
    temperature=0.7)
print(response[0]['generated_text'])

It allows the model to focus on different parts of the input sequence simultaneously by using multiple sets of weights.


# Different paragraph

In [2]:
paragraph = """
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers.
The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network.
We employ a residual connection around each of the two sub-layers, followed by layer normalization.
That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself.
To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension 512.
Decoder: The decoder is also composed of a stack of N = 6 identical layers.
In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack.
Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization.
We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions.
This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i.
Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors.
The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.
"""

In [22]:
from transformers import pipeline

summarizer = pipeline("summarization", model="google/pegasus-xsum")
# summary = summarizer(paragraph,
#                      max_length=40,      # Pegasus performs better when forced to be concise but not 'tiny'
#                      min_length=15,      # Forces it to explain more than just a headline
#                      num_beams=50,        # High beam count helps find more accurate logical paths
#                      )

summary = summarizer(paragraph,
                     max_length=60,      # Pegasus performs better when forced to be concise but not 'tiny'
                     min_length=20)

print(summary[0]['summary_text'])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


In this paper, we present two encoders and two decoders, each with its own self-attention mechanism.
